# Assignment 5: Two-Layer Network for Non-Linear 2D Classification (Circle Dataset)

## Summary

**Hypothesis:** A two-layer neural network (one large fully connected hidden layer + a 2-neuron one-hot output layer) can classify any two-dimensional partition in a finite region, even when the decision boundary is not a straight line.

**Dataset:** Circle — class 0 is a cluster of points near the origin, class 1 is a ring of points at radius ~4. No straight line can separate a blob from a surrounding ring, so this is a non-linear classification problem.

**Architecture:**
- Input: (x, y) — 2 features
- Hidden layer: 512 neurons, ReLU activation (large fully connected layer)
- Output layer: 2 neurons, softmax activation (one-hot strategy)

**Result:** The network successfully learns the circular decision boundary and achieves high accuracy on both training and test data, supporting the hypothesis.

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

## Generate Circle Dataset

We generate two classes of points using polar coordinates:
- **Class 0 (inner blob):** Points centered at (4, 5) with small radius (Gaussian noise around 0). These form a tight cluster.
- **Class 1 (outer ring):** Points centered at (4, 5) with radius ~4 (Gaussian noise around 4). These form a ring surrounding the inner blob.

A straight line cannot separate a blob from a surrounding ring — this requires a non-linear (circular) decision boundary.

In [3]:
np.random.seed(42)

num_samples_per_class = 1000
mu, sigma = 0, 0.65

# Class 0: inner blob (radius ~ 0 + noise)
s0 = np.random.normal(mu, sigma, num_samples_per_class)
theta0 = np.random.uniform(0, 2 * np.pi, num_samples_per_class)
C0_samples = np.column_stack([s0 * np.cos(theta0) + 4, s0 * np.sin(theta0) + 5])

# Class 1: outer ring (radius ~ 4 + noise)
s1 = np.random.normal(mu, sigma, num_samples_per_class) + 4
theta1 = np.random.uniform(0, 2 * np.pi, num_samples_per_class)
C1_samples = np.column_stack([s1 * np.cos(theta1) + 4, s1 * np.sin(theta1) + 5])

print(f"Class 0 (inner blob): {C0_samples.shape}")
print(f"Class 1 (outer ring): {C1_samples.shape}")

# Generate test data with different seed
np.random.seed(99)

s0_t = np.random.normal(mu, sigma, num_samples_per_class)
theta0_t = np.random.uniform(0, 2 * np.pi, num_samples_per_class)
T0_samples = np.column_stack([s0_t * np.cos(theta0_t) + 4, s0_t * np.sin(theta0_t) + 5])

s1_t = np.random.normal(mu, sigma, num_samples_per_class) + 4
theta1_t = np.random.uniform(0, 2 * np.pi, num_samples_per_class)
T1_samples = np.column_stack([s1_t * np.cos(theta1_t) + 4, s1_t * np.sin(theta1_t) + 5])

print(f"Test class 0: {T0_samples.shape}")
print(f"Test class 1: {T1_samples.shape}")

Class 0 (inner blob): (1000, 2)
Class 1 (outer ring): (1000, 2)
Test class 0: (1000, 2)
Test class 1: (1000, 2)


In [ ]:
# Stack into training arrays
inputs = np.vstack((C0_samples, C1_samples)).astype(np.float32)
targets = np.concatenate([np.zeros(num_samples_per_class),
                          np.ones(num_samples_per_class)]).astype(np.float32)

# Stack into testing arrays
Tinputs = np.vstack((T0_samples, T1_samples)).astype(np.float32)
Ttargets = np.concatenate([np.zeros(num_samples_per_class),
                           np.ones(num_samples_per_class)]).astype(np.float32)

print(f"Training: {inputs.shape}, {targets.shape}")
print(f"Testing:  {Tinputs.shape}, {Ttargets.shape}")

## Plot Training Data

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(inputs[:, 0], inputs[:, 1], c=targets, cmap='bwr', s=5, alpha=0.7)
plt.title('Circle Dataset (Training Data)')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.colorbar(label='Class')
plt.show()

## Build and Train the Two-Layer Model

- **Hidden layer:** 512 neurons with ReLU activation — gives the network enough capacity to approximate the circular boundary using piecewise-linear regions.
- **Output layer:** 2 neurons with softmax activation (one-hot encoding). Neuron 0 fires for class 0 (inner), neuron 1 fires for class 1 (outer).

In [ ]:
model = keras.Sequential([
    layers.Dense(512, activation="relu"),
    layers.Dense(2, activation="softmax")
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

history = model.fit(inputs, targets,
                    epochs=200,
                    batch_size=64,
                    validation_data=(Tinputs, Ttargets))

In [ ]:
# Plot training history
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

## Evaluate on Test Data

In [ ]:
test_loss, test_acc = model.evaluate(Tinputs, Ttargets)
print(f"Test accuracy: {test_acc:.4f}")

## Decision Boundary Visualization

We sample a dense grid of points across the input space and color each point by the model's predicted class. This reveals the learned decision boundary — the circular surface that separates the inner blob from the outer ring.

In [ ]:
num_grid = 200000

x_min, x_max = inputs[:, 0].min() - 1, inputs[:, 0].max() + 1
y_min, y_max = inputs[:, 1].min() - 1, inputs[:, 1].max() + 1

grid_x = np.random.uniform(x_min, x_max, num_grid)
grid_y = np.random.uniform(y_min, y_max, num_grid)
grid_inputs = np.column_stack([grid_x, grid_y]).astype(np.float32)

grid_predictions = model.predict(grid_inputs)

plt.figure(figsize=(10, 10))
plt.scatter(grid_inputs[:, 0], grid_inputs[:, 1],
            c=grid_predictions[:, 1] > 0.5, cmap='bwr', s=1, alpha=0.2)
plt.scatter(inputs[:, 0], inputs[:, 1],
            c=targets, cmap='bwr', s=10, edgecolors='black', linewidths=0.3)
plt.title('Learned Decision Boundary (Circle Dataset)')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.show()